# Project 4 — Deep Learning Systems

## Problem Definition

This project uses deep learning with a Transformer model to detect risky language in rental listing text.

The dataset used is Toronto Kijiji Rental Data from Kaggle. It contains real rental ads with text descriptions and rental information.

The goal of this project is to classify listing text into categories such as Normal, Moderate Risk, or High Risk based on language patterns.

## Dataset Source

Toronto Kijiji Rental Data (Kaggle)

https://www.kaggle.com/datasets/umeshkhatiwada/toronto-kijiji-rental-data

In [2]:
import pandas as pd

# Load dataset file
df = pd.read_csv("kijiji_rental_ads_4106.csv")

# Show number of rows and columns
print("Rows and Columns:", df.shape)

# Show all column names
print("\nColumns:")
print(df.columns)

# Show first 5 rows
df.head()

Rows and Columns: (4106, 23)

Columns:
Index(['Title', 'Price($)', 'Address', 'Date Posted', 'Building Type',
       'Bedrooms', 'Bathrooms', 'Utilities', 'Wi-Fi and More',
       'Parking Included', 'Agreement Type', 'Move-In Date', 'Pet Friendly',
       'Size (sqft)', 'Furnished', 'Air Conditioning',
       'Personal Outdoor Space', 'Smoking Permitted', 'Appliances',
       'Amenities', 'Description', 'Visit Counter', 'url'],
      dtype='object')


,Title,Price($),Address,Date Posted,Building Type,Bedrooms,Bathrooms,Utilities,Wi-Fi and More,Parking Included,...,Size (sqft),Furnished,Air Conditioning,Personal Outdoor Space,Smoking Permitted,Appliances,Amenities,Description,Visit Counter,url
0,6020 Bathurst Street - Valencia Towers Apartme...,3209.0,"6020 Bathurst Street, Toronto, ON, M2R 1Z8",2024-02-24 23:30:00,Apartment,2,1,NaN,Not Included,0,...,912,No,No,Not Included,No,Fridge / Freezer,Elevator in Building,Valencia Towers is a student- and family-frien...,NaN,https://www.kijiji.ca/v-apartments-condos/city...
1,RENOVATED BACHELOR SUITE AVAILABLE! Lakeview ...,2000.0,"22 Close Avenue, Toronto, ON, M6K 2V2",2024-03-14 00:09:49,Apartment,Bachelor/Studio,1,"Hydro_No,Heat_No,Water_Yes",Not Included,0,...,445,No,No,Balcony,No,"Laundry (In Building), Fridge / Freezer","Gym, Pool, Storage Space, Elevator in Building","Bachelors, 1 Bath, Recently Renovated Kitchen ...",NaN,https://www.kijiji.ca/v-apartments-condos/city...
2,50 Driftwood - Ruby Heights Apartment for Rent,2819.0,"50 Driftwood, Toronto, ON, M3N 2M6",2024-03-03 00:08:58,Apartment,2,1,NaN,Not Included,0,...,904,No,No,Not Included,No,Fridge / Freezer,"Storage Space, Elevator in Building","Ruby Heights, located in the North York distri...",NaN,https://www.kijiji.ca/v-apartments-condos/city...
3,1 Bed Apartment Rent today,2519.0,"100 Parkway Forest Drive, Toronto, ON, M2J 1L6",2024-03-06 00:35:06,Apartment,1,1,"Hydro_No,Heat_Yes,Water_Yes",Not Included,0,...,665,No,No,Yard,Yes,"Laundry (In Building), Fridge / Freezer","Gym, Pool, 24 Hour Security, Storage Space, El...","For a limited time, you can receive ONE MONTH ...",NaN,https://www.kijiji.ca/v-apartments-condos/city...
4,Brand New 2-bedroom Rental in North York! Yor...,2690.0,"1225 York Road, Toronto, ON, M3A 1Y4",2024-03-04 00:39:41,Apartment,2,1,NaN,Not Included,0,...,Not Available,No,Yes,Balcony,Yes,"Laundry (In Unit), Dishwasher, Fridge / Freezer","Gym, Bicycle Parking, Storage Space, Elevator ...",Realstar's ONE225 York Mills is North Yorks ne...,NaN,https://www.kijiji.ca/v-apartments-condos/city...


## Dataset Inspection Notes

The dataset contains 4,106 rows and 23 columns, which is enough for a deep learning text classification project.

Important columns in this dataset include `Title`, `Price($)`, `Bedrooms`, `Bathrooms`, `Address`, and `Description`.

The most important text column for this project is `Description`, because it contains the main rental listing language that can be used for Transformer-based NLP.

Some columns may need cleaning because they contain missing values, mixed formats, or text that is not useful for the classification task.

In [4]:
# Check data types
print("Data types:")
print(df.dtypes)

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# Check the main text column
print("\nExample descriptions:")
for i, text in enumerate(df["Description"].dropna().head(3), 1):
    print(f"\nExample {i}:")
    print(text[:500])  # show first 500 characters only

Data types:
Title                      object
Price($)                  float64
Address                    object
Date Posted                object
Building Type              object
Bedrooms                   object
Bathrooms                  object
Utilities                  object
Wi-Fi and More             object
Parking Included           object
Agreement Type             object
Move-In Date               object
Pet Friendly               object
Size (sqft)                object
Furnished                  object
Air Conditioning           object
Personal Outdoor Space     object
Smoking Permitted          object
Appliances                 object
Amenities                  object
Description                object
Visit Counter              object
url                        object
dtype: object

Missing values:
Title                        7
Price($)                   256
Address                     12
Date Posted                608
Building Type             1025
Bedrooms            

## Dataset Inspection Notes

- Most columns in the dataset are stored in text format, while the rental price column is numeric.

- Several columns contain missing values, especially optional property features such as amenities, utilities, and move-in date.

- The most important column for this project is `Description`, because it contains the main rental listing text that will be used for Transformer-based classification.

- The example descriptions show real rental language, promotional wording, contact requests, and housing details. This makes the dataset suitable for natural language processing.

- Before modeling, rows with missing descriptions will be removed and text data will be cleaned.

## Data Cleaning and Text Preparation

The dataset was cleaned before deep learning modeling.

The main text column used for this project is `Description`.

Rows with missing descriptions were removed, duplicate text entries were removed, and extra spaces were cleaned from the text.

In [5]:
# Make a clean copy of the dataset
df_clean = df.copy()

# Keep only rows that have description text
df_clean = df_clean.dropna(subset=["Description"])

# Remove duplicate descriptions
df_clean = df_clean.drop_duplicates(subset=["Description"])

# Convert description to string
df_clean["Description"] = df_clean["Description"].astype(str)

# Remove extra spaces
df_clean["Description"] = df_clean["Description"].str.replace(r"\s+", " ", regex=True).str.strip()

# Show updated shape
print("Rows and Columns after cleaning:", df_clean.shape)

# Show first few cleaned rows
df_clean[["Title", "Description"]].head()

Rows and Columns after cleaning: (2796, 23)


,Title,Description
0,6020 Bathurst Street - Valencia Towers Apartme...,Valencia Towers is a student- and family-frien...
1,RENOVATED BACHELOR SUITE AVAILABLE! Lakeview ...,"Bachelors, 1 Bath, Recently Renovated Kitchen ..."
2,50 Driftwood - Ruby Heights Apartment for Rent,"Ruby Heights, located in the North York distri..."
3,1 Bed Apartment Rent today,"For a limited time, you can receive ONE MONTH ..."
4,Brand New 2-bedroom Rental in North York! Yor...,Realstar's ONE225 York Mills is North Yorks ne...
